# 00 - How the IPEDS API really behaves

The Urban Institute's Education Data Portal is a free REST API over IPEDS.
No key, no signup. This notebook is the five things that will bite you if
you skip them - each one verified against the live API, not the docs.

Run the cells top to bottom. Nothing here takes more than a few seconds.

In [1]:
import sys; sys.path.insert(0, "..")   # so `from src...` finds the project code
import requests
BASE = "https://educationdata.urban.org/api/v1/college-university/ipeds"

r = requests.get(f"{BASE}/directory/2016/?fips=53").json()
print(r.keys())
print("count:", r["count"], "| rows returned:", len(r["results"]), "| next page:", r["next"])

dict_keys(['count', 'next', 'previous', 'results'])
count: 129 | rows returned: 129 | next page: None


## 1. Pages hold 10,000 rows - follow `next`, never build `?page=N`

A small query comes back complete, with `next = None`. A big one does not.

In [2]:
big = requests.get(f"{BASE}/grad-rates/2016/").json()
print("total rows available:", f"{big['count']:,}")
print("rows in this page:   ", f"{len(big['results']):,}")
print("next:", big["next"])

# Asking for a page past the end is a 404, not an empty list.
print("page 999 ->", requests.get(f"{BASE}/grad-rates/2016/?page=999").status_code)

total rows available: 224,501
rows in this page:    10,000
next: https://educationdata.urban.org/api/v1/college-university/ipeds/grad-rates/2016/?page=2


page 999 -> 404


So the loop is always *"start at the URL, follow `next` until it is None"*.
That is exactly what `src.data_pull.fetch()` does, and it caches the result
to `data/` as parquet so the second run costs nothing.

In [3]:
from src.data_pull import fetch
wa = fetch("directory", 2016, fips=53)      # first run: hits the API
wa = fetch("directory", 2016, fips=53)      # second run: reads the parquet
print(wa.shape)
wa[["unitid", "inst_name", "sector", "state_abbr"]].head()

(129, 96)


,unitid,inst_name,sector,state_abbr
0,102845,Charter College,3,WA
1,234401,Beauty Academy,9,WA
2,234492,The Art Institute of Seattle,3,WA
3,234650,Evergreen Beauty and Barber College-Bellevue,9,WA
4,234669,Bellevue College,1,WA


## 2. Filter on the server - it is the single biggest speed-up

`grad-rates` is one row per institution x race x sex x subcohort. Asking for
the race and sex *totals* turns 224,501 rows (23 pages) into 9,118 (one page).

In [4]:
for q in ["", "?race=99&sex=99"]:
    print(f"{q or '(no filter)':20s} ->", f"{requests.get(f'{BASE}/grad-rates/2016/{q}').json()['count']:>8,} rows")

(no filter)          ->  224,501 rows


?race=99&sex=99      ->    9,118 rows


## 3. Missing values are -1, -2, -3 - not null

This is the one that silently ruins a regression. A "graduation rate" of
-1.00 is not a graduation rate.

In [5]:
g = fetch("grad-rates", 2016, race=99, sex=99)
print(g.completion_rate_150pct.describe()[["min", "max"]])
print("rows with a sentinel value:", (g.completion_rate_150pct < 0).sum())

from src.data_pull import to_na
clean = to_na(g, ["completion_rate_150pct"])
print("after to_na, min is", clean.completion_rate_150pct.min())

min   -1.0
max    1.0
Name: completion_rate_150pct, dtype: float64
rows with a sentinel value: 205
after to_na, min is 0.0


`to_na` is deliberately scoped to the columns you name. A blanket
"replace all -2 with NaN" would destroy `subcohort` and `fed_aid_type`,
where -2 is a real category.

## 4. Coverage is uneven - check before you promise a 20-year panel

In [6]:
from src.data_pull import COVERAGE
for topic, years in COVERAGE.items():
    print(f"{topic:26s} {min(years)}-{max(years)}")

directory                  2003-2023
finance                    2003-2017
sfa-all-undergraduates     2004-2021
grad-rates                 2003-2023
grad-rates-pell            2015-2023


In [7]:
# `finance` really does stop. This is a coverage gap, not a bad request:
for y in [2016, 2017, 2018, 2019]:
    print(y, requests.get(f"{BASE}/finance/{y}/").json()["count"], "rows")

2016 7153 rows


2017 6857 rows
2018 0 rows


2019 0 rows


That single fact sets the study period for anything using institutional
finances: **2003-2017**, no matter how recent the other tables go.

## 5. Now pull everything

`python osg/pull_all_panels.py` downloads all five topics for every year they
exist (about 20-40 minutes, fully cached, safe to interrupt and restart).
After that, notebooks 01-03 run offline in seconds.